# W1C2 Lab: Splitting Text into Pieces

Run every cell from the top. **Everything already works.**

Today you will:

1. Split text into words three different ways and see where they disagree.
2. Pull structured data out of messy text with a regular expression.
3. Measure how close two words are with edit distance.

There is no test to run and nothing to submit. Each task tells you what
you should see when it is right.

In [ ]:
# Setup.
import re
import nltk
import pandas as pd
from nltk.tokenize import word_tokenize

# nltk ships code, not data. This fetches the sentence splitter the first
# time and is instant afterwards. (Or: uv run python scripts/setup_data.py)
nltk.download("punkt_tab", quiet=True)

MESSY = "Dr. O'Neill emailed prof@trinity.edu on 09/12/2026 -- it wasn't cheap ($42.50)!"
print(MESSY)

## Part 1. Three ways to split a sentence

Tokenizing looks trivial until you try it. Compare the simplest possible
method with a real one.

In [ ]:
# GIVEN. Three tokenizers on the same messy sentence.
by_space = MESSY.split()
by_regex = re.findall(r"[A-Za-z']+", MESSY)
by_nltk  = word_tokenize(MESSY)

table = pd.DataFrame({
    "method": ["split()", "regex letters", "nltk"],
    "n_tokens": [len(by_space), len(by_regex), len(by_nltk)],
    "first 6": [str(by_space[:6]), str(by_regex[:6]), str(by_nltk[:6])],
})
print(table.to_string(index=False))
print()
print("Notice: split() keeps '($42.50)!' glued together as one token.")

In [ ]:
# ================== YOUR TURN 1 ==================
# The regex above throws away every number, so the price and the date
# vanish. Widen it to keep digits, dots and the dollar sign too.
#
# Hint: add 0-9 and the characters you want inside the [ ] group
#
# Expected: '42.50' or '$42.50' appears in the token list, and n_tokens goes up.
# ===============================================
PATTERN = r"[A-Za-z']+"          # <-- widen this

tokens = re.findall(PATTERN, MESSY)
print(tokens)
print()
kept_a_number = any(any(ch.isdigit() for ch in t) for t in tokens)
print("kept a number:", kept_a_number)

## Part 2. Pulling facts out of text

A regex is how you get structure out of unstructured text: emails, dates,
prices, phone numbers. One pattern per thing you want.

In [ ]:
# GIVEN. Two patterns that already work.
EMAIL = r"[\w.]+@[\w.]+\.\w+"
PRICE = r"\$\d+(?:\.\d{2})?"

print("emails:", re.findall(EMAIL, MESSY))
print("prices:", re.findall(PRICE, MESSY))

In [ ]:
# ================== YOUR TURN 2 ==================
# Write a pattern that finds the date 09/12/2026.
#
# Hint: \d{2} matches exactly two digits; escape the slash or just write it
#
# Expected: ['09/12/2026']
# ===============================================
DATE = r"NOTHING_YET"          # <-- write the pattern

found = re.findall(DATE, MESSY)
print("dates:", found)
print("correct:", found == ["09/12/2026"])

## Part 3. How different are two words?

Edit distance counts the single-character edits needed to turn one word
into another. It is how spell-checkers rank suggestions.

In [ ]:
# GIVEN. Distance from a misspelling to some candidates.
misspelled = "aceptable"
candidates = ["acceptable", "acceptably", "accessible", "table", "cable"]

rows = [(c, nltk.edit_distance(misspelled, c)) for c in candidates]
scores = pd.DataFrame(rows, columns=["candidate", "edits"]).sort_values("edits")
print(scores.to_string(index=False))
print()
print("best guess:", scores.iloc[0]["candidate"])

In [ ]:
# ================== YOUR TURN 3 ==================
# Pick your own misspelling and see whether edit distance finds the
# word you meant. Try a word where it FAILS, for example one where you
# typed two letters wrong.
#
# Expected: the closest candidate has the smallest number of edits. It is not
#           always the word you meant, which is why real spell-checkers also use
#           how common each word is.
# ===============================================
MY_TYPO = "aceptable"          # <-- change me

rows = [(c, nltk.edit_distance(MY_TYPO, c)) for c in candidates]
out = pd.DataFrame(rows, columns=["candidate", "edits"]).sort_values("edits")
print(out.to_string(index=False))
print()
print("closest:", out.iloc[0]["candidate"], "with", out.iloc[0]["edits"], "edits")

## Answers

Try each task before reading.

In [ ]:
# YOUR TURN 1
#   PATTERN = r"[A-Za-z'$0-9.]+"
#   Widening a regex is a trade: this one now also keeps the trailing "." of
#   "Dr." as part of the token. There is no perfect pattern, which is the point.
#
# YOUR TURN 2
#   DATE = r"\d{2}/\d{2}/\d{4}"
#
# YOUR TURN 3
#   Any typo works. Edit distance treats every edit as equally likely, so
#   "cable" and "table" score well against short typos even though no one meant
#   them. Real correctors combine distance with word frequency.